In [2]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":      
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_CSV = PROJECT_ROOT / "data" / "raw" / "data.csv"
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "news_tanpa_embedding.parquet"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "news_bersih.parquet"

log = {}   

In [3]:
if not RAW_CSV.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {RAW_CSV}. "
        "Pastikan Milestone 1 sudah menempatkan data di data/raw/."
    )

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    lf = pl.scan_csv(RAW_CSV, infer_schema_length=0)
    kolom_dipakai = [c for c in lf.collect_schema().names() if c != "embedding"]
    lf.select(kolom_dipakai).sink_parquet(DATA_PATH)

print(f"CSV mentah : {RAW_CSV.stat().st_size / 1024**2:,.1f} MiB")
print(f"Parquet    : {DATA_PATH.stat().st_size / 1024**2:,.1f} MiB")

CSV mentah : 703.7 MiB
Parquet    : 30.1 MiB


In [4]:
df = pl.read_parquet(DATA_PATH)
log["awal"] = len(df)
print(f"Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom")
df.head()

Ukuran awal: 32,735 baris x 10 kolom


id,source,title,image,url,content,date,created_at,updated_at,summary
str,str,str,str,str,str,str,str,str,str
"""83""","""tempo""","""Depo Plumpang Terbakar, Anggot…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Anggota Ko…","""2023-03-04 06:18:13+00""","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Anggota Komisi VII DPR RI Rofi…"
"""84""","""tempo""","""Jokowi Perintahkan Wapres Ma'r…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Presiden J…","""2023-03-04 06:04:38+00""","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Presiden Joko Widodo telah mem…"
"""85""","""tempo""","""HNW Mendukung Jamaah Umroh Fir…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""INFO NASIONAL - Wakil Ketua MP…","""2023-03-04 06:18:04+00""","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Wakil Ketua MPR RI Dr. H. M. H…"
"""86""","""tempo""","""Tim Dokkes Polri Telah Terima …","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Tim Kedokt…","""2023-03-04 06:44:10+00""","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Tim Kedokteran dan Kesehatan (…"
"""87""","""tempo""","""Bamsoet Ajak Komunitas Otomoti…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""INFO NASIONAL - Ketua MPR RI s…","""2023-03-04 06:38:57+00""","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Ketua MPR RI Bambang Soesatyo …"


In [5]:
tidak_utc = df.filter(~pl.col("date").str.ends_with("+00")).height
print(f"Baris dengan offset selain +00: {tidak_utc:,}")

df = df.with_columns(
    pl.col("id").cast(pl.Int64, strict=False),
    pl.col("date").str.replace(r"\+00$", "").str.to_datetime(strict=False).alias("published_at"),
    pl.col("created_at").str.to_datetime(strict=False),
    pl.col("updated_at").str.to_datetime(strict=False),
).drop("date")

cek = df.select(pl.col("id", "published_at", "created_at", "updated_at").null_count())
print(cek)
assert cek.sum_horizontal().item() == 0, "Ada nilai gagal di-parse; periksa format tanggal/id."
df.schema

Baris dengan offset selain +00: 0
shape: (1, 4)
┌─────┬──────────────┬────────────┬────────────┐
│ id  ┆ published_at ┆ created_at ┆ updated_at │
│ --- ┆ ---          ┆ ---        ┆ ---        │
│ u32 ┆ u32          ┆ u32        ┆ u32        │
╞═════╪══════════════╪════════════╪════════════╡
│ 0   ┆ 0            ┆ 0          ┆ 0          │
└─────┴──────────────┴────────────┴────────────┘


Schema([('id', Int64),
        ('source', String),
        ('title', String),
        ('image', String),
        ('url', String),
        ('content', String),
        ('created_at', Datetime(time_unit='us', time_zone=None)),
        ('updated_at', Datetime(time_unit='us', time_zone=None)),
        ('summary', String),
        ('published_at', Datetime(time_unit='us', time_zone=None))])

In [6]:
connection = duckdb.connect()
connection.register("dataset", df)

profil = connection.sql("SUMMARIZE SELECT * FROM dataset").pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""id""","""BIGINT""","""33""","""63567""",32590,"""34081.74877042921""","""18044.068638414126""","""17745""","""35146""","""49818""",32735,0.00
"""source""","""VARCHAR""","""cnbcindonesia""","""tempo""",7,null,null,null,null,null,32735,0.00
"""title""","""VARCHAR""",""" Daftar Panglima Kostrad Dari …","""​​Tips Atasi Masalah Jerawat d…",32823,null,null,null,null,null,32735,0.00
"""image""","""VARCHAR""","""""","""https://statik.tempo.co/data/2…",26682,null,null,null,null,null,32735,0.00
"""url""","""VARCHAR""","""https://bisnis.tempo.co/read/1…","""https://www.suara.com/news/202…",34771,null,null,null,null,null,32735,0.00
"""content""","""VARCHAR""","""""","""zakat terdapat di dalam rukun …",32550,null,null,null,null,null,32735,0.00
"""created_at""","""TIMESTAMP""","""2023-03-03 16:08:03.144472""","""2023-04-11 20:30:13.315332""",3481,"""2023-03-27 01:04:27.096193""",null,"""2023-03-18 07:33:36.430347""","""2023-03-27 00:00:31.300828""","""2023-04-04 21:14:24.632886""",32735,0.00
"""updated_at""","""TIMESTAMP""","""2023-03-03 16:08:03.144472""","""2023-04-11 20:30:13.315332""",3481,"""2023-03-27 01:04:27.096193""",null,"""2023-03-18 07:33:36.430347""","""2023-03-27 00:00:31.300828""","""2023-04-04 21:14:24.632886""",32735,0.00
"""summary""","""VARCHAR""","""""AS Kembangkan Vaksin Oral Cov…","""ular di ruang kargo. Pesawat m…",33448,null,null,null,null,null,32735,0.03


In [7]:
jumlah_baris = len(df)
KOLOM_TEKS = ["title", "image", "content", "summary"]

kosong = df.select([(pl.col(c).str.strip_chars() == "").sum().alias(c) for c in KOLOM_TEKS])
print("String kosong per kolom:")
print(kosong)

null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns((pl.col("jumlah_null") / jumlah_baris * 100).round(2).alias("persen_null"))
    .filter(pl.col("jumlah_null") > 0)
    .sort("jumlah_null", descending=True)
)
null_summary

String kosong per kolom:
shape: (1, 4)
┌───────┬───────┬─────────┬─────────┐
│ title ┆ image ┆ content ┆ summary │
│ ---   ┆ ---   ┆ ---     ┆ ---     │
│ u32   ┆ u32   ┆ u32     ┆ u32     │
╞═══════╪═══════╪═════════╪═════════╡
│ 0     ┆ 160   ┆ 441     ┆ 0       │
└───────┴───────┴─────────┴─────────┘


kolom,jumlah_null,persen_null
str,u32,f64
"""summary""",10,0.03


In [8]:
df_missing_handled = (
    df.with_columns([pl.col(c).str.strip_chars().alias(c) for c in KOLOM_TEKS])
      .with_columns([
          pl.when(pl.col(c) == "").then(None).otherwise(pl.col(c)).alias(c) for c in KOLOM_TEKS
      ])
      .with_columns(
          pl.col("content").is_null().alias("content_kosong"),
          pl.col("summary").is_null().alias("summary_kosong"),
          pl.col("image").is_null().alias("image_kosong"),
      )
)

kolom_asli = df.columns
print(f"Null sebelum: {df.null_count().sum_horizontal().sum():,}")
print(f"Null sesudah: {df_missing_handled.select(kolom_asli).null_count().sum_horizontal().sum():,}")
print(df_missing_handled.select(
    pl.col("content_kosong").sum(), pl.col("summary_kosong").sum(), pl.col("image_kosong").sum()
))
log["setelah_missing"] = len(df_missing_handled)

Null sebelum: 10
Null sesudah: 611
shape: (1, 3)
┌────────────────┬────────────────┬──────────────┐
│ content_kosong ┆ summary_kosong ┆ image_kosong │
│ ---            ┆ ---            ┆ ---          │
│ u32            ┆ u32            ┆ u32          │
╞════════════════╪════════════════╪══════════════╡
│ 441            ┆ 10             ┆ 160          │
└────────────────┴────────────────┴──────────────┘


In [9]:
KOLOM_NUMERIK = "content_len"
BATAS_MIN_DOMAIN = 100

df_missing_handled = df_missing_handled.with_columns(
    pl.col("content").str.len_chars().cast(pl.Int64).alias(KOLOM_NUMERIK)
)

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"Batas bawah: {batas_bawah}, Batas atas: {batas_atas}")

lower = int(max(batas_bawah, BATAS_MIN_DOMAIN))
upper = int(batas_atas)

df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=lower, upper_bound=upper).alias("content_len_clip"),
    (pl.col(KOLOM_NUMERIK) > upper).fill_null(False).alias("content_terlalu_panjang"),
    (pl.col(KOLOM_NUMERIK) < BATAS_MIN_DOMAIN).fill_null(False).alias("content_pendek"),
)

print(df_outlier_handled.select(
    pl.col("content_terlalu_panjang").sum().alias("terlalu_panjang"),
    pl.col("content_pendek").sum().alias("pendek (tidak termasuk kosong)"),
))
log["setelah_outlier"] = len(df_outlier_handled)

Q1=1536.0, Q3=2789.0, IQR=1253.0
Batas bawah: -343.5, Batas atas: 4668.5
shape: (1, 2)
┌─────────────────┬────────────────────────────────┐
│ terlalu_panjang ┆ pendek (tidak termasuk kosong) │
│ ---             ┆ ---                            │
│ u32             ┆ u32                            │
╞═════════════════╪════════════════════════════════╡
│ 1310            ┆ 2                              │
└─────────────────┴────────────────────────────────┘


In [10]:
print(df_outlier_handled.select(
    pl.col("published_at").min().alias("terbit_min"),
    pl.col("published_at").max().alias("terbit_max"),
))

per_tahun = (
    df_outlier_handled.group_by(pl.col("published_at").dt.year().alias("tahun"))
    .agg(pl.len().alias("n")).sort("tahun")
)
print(per_tahun)

setelah_crawl = df_outlier_handled.filter(
    pl.col("published_at") > pl.col("created_at") + pl.duration(days=1)
).height
print(f"Terbit setelah waktu pengumpulan (+1 hari): {setelah_crawl:,}")

shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ terbit_min          ┆ terbit_max          │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2018-08-10 05:11:58 ┆ 2023-04-11 20:13:25 │
└─────────────────────┴─────────────────────┘
shape: (2, 2)
┌───────┬───────┐
│ tahun ┆ n     │
│ ---   ┆ ---   │
│ i32   ┆ u32   │
╞═══════╪═══════╡
│ 2018  ┆ 1     │
│ 2023  ┆ 32734 │
└───────┴───────┘
Terbit setelah waktu pengumpulan (+1 hari): 0


In [11]:
KOLOM_KUNCI = "id"

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique()
print(f"Total baris: {total_baris:,}")
print(f"Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}")
print(f"Duplikat {KOLOM_KUNCI}: {total_baris - baris_unik:,}")
print(f"Duplikat url: {total_baris - df_outlier_handled['url'].n_unique():,}")

df_dedup_kunci = (
    df_outlier_handled
    .unique(subset=[KOLOM_KUNCI], keep="first", maintain_order=True)
    .unique(subset=["url"], keep="first", maintain_order=True)
)

judul_kembar = (
    df_dedup_kunci.group_by("title")
    .agg(
        pl.len().alias("n"),
        pl.col("source").n_unique().alias("n_source"),
        pl.col("content").n_unique().alias("n_content_berbeda"),
    )
    .filter(pl.col("n") > 1)
    .sort("n", descending=True)
)
print(f"Judul yang muncul lebih dari sekali: {len(judul_kembar):,}")
judul_kembar.head(10)

Total baris: 32,735
Baris unik berdasarkan id: 32,735
Duplikat id: 0
Duplikat url: 0
Judul yang muncul lebih dari sekali: 80


title,n,n_source,n_content_berbeda
str,u32,u32,u32
"""Pegadaian Borong Empat Penghar…",3,3,3
"""Pertamina Geothermal Energy Bu…",3,3,3
"""Menaker: THR Tidak Boleh Dicic…",3,2,2
"""Akademisi Nilai Program Food E…",3,3,3
"""Silicon Valley Bank Bangkrut, …",2,2,2
"""Pemerintah Ingin Penerima KUR …",2,2,2
"""Gempa 5,1 Magnitudo Guncang Me…",2,1,2
"""Premi Risiko Investasi Credit …",2,1,2
"""Gempa 4,5 Magnitudo Guncang Bo…",2,1,2


In [12]:
punya_konten = df_dedup_kunci.filter(pl.col("content").is_not_null())
tanpa_konten = df_dedup_kunci.filter(pl.col("content").is_null())

punya_konten = punya_konten.unique(subset=["title", "content"], keep="first", maintain_order=True)
df_dedup = pl.concat([punya_konten, tanpa_konten]).sort(KOLOM_KUNCI)

print(f"Dihapus (title+content sama): {len(df_dedup_kunci) - len(df_dedup):,}")
print(f"Baris setelah deduplikasi   : {len(df_dedup):,}")
log["setelah_dedup"] = len(df_dedup)

Dihapus (title+content sama): 10
Baris setelah deduplikasi   : 32,725


In [13]:
connection.register("dataset_bersih", df_dedup)

validasi = connection.sql(f"""
    SELECT
        COUNT(*)                                             AS total_baris,
        COUNT(*) - COUNT(DISTINCT id)                        AS pelanggaran_id_duplikat,
        SUM(CASE WHEN url IS NULL OR NOT starts_with(url, 'http') THEN 1 ELSE 0 END)
                                                             AS pelanggaran_url,
        SUM(CASE WHEN source IS NULL OR source = '' THEN 1 ELSE 0 END)
                                                             AS pelanggaran_source,
        SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END)       AS pelanggaran_title,
        SUM(CASE WHEN published_at IS NULL
                   OR published_at > created_at + INTERVAL 1 DAY THEN 1 ELSE 0 END)
                                                             AS pelanggaran_tanggal_terbit,
        SUM(CASE WHEN created_at > updated_at THEN 1 ELSE 0 END)
                                                             AS pelanggaran_urutan_waktu,
        SUM(CASE WHEN content_len IS NOT NULL AND content_len < {BATAS_MIN_DOMAIN}
                 THEN 1 ELSE 0 END)                          AS pelanggaran_konten_pendek
    FROM dataset_bersih
""").pl()

validasi.transpose(include_header=True, header_name="aturan", column_names=["jumlah"])

aturan,jumlah
str,"decimal[38,0]"
"""total_baris""",32725
"""pelanggaran_id_duplikat""",0
"""pelanggaran_url""",0
"""pelanggaran_source""",0
"""pelanggaran_title""",0
"""pelanggaran_tanggal_terbit""",0
"""pelanggaran_urutan_waktu""",0
"""pelanggaran_konten_pendek""",2


In [14]:
print(df_dedup.group_by("source").agg(pl.len().alias("n")).sort("n", descending=True))

shape: (7, 2)
┌───────────────┬──────┐
│ source        ┆ n    │
│ ---           ┆ ---  │
│ str           ┆ u32  │
╞═══════════════╪══════╡
│ kumparan      ┆ 8578 │
│ okezone       ┆ 7066 │
│ cnbcindonesia ┆ 4690 │
│ suara         ┆ 4004 │
│ cnnindonesia  ┆ 3735 │
│ tempo         ┆ 3641 │
│ jawapos       ┆ 1011 │
└───────────────┴──────┘


In [16]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f"Baris sebelum cleaning: {len(df):,}")
print(f"Baris sesudah cleaning: {len(df_dedup):,}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")
pl.DataFrame({"tahap": list(log.keys()), "jumlah_baris": list(log.values())})

Baris sebelum cleaning: 32,735
Baris sesudah cleaning: 32,725
Data bersih tersimpan di: /home/jovyan/work/data/processed/news_bersih.parquet


tahap,jumlah_baris
str,i64
"""awal""",32735
"""setelah_missing""",32735
"""setelah_outlier""",32735
"""setelah_dedup""",32725


## Catatan Milestone 2
- **Dimensi kualitas paling bermasalah:** (completeness `content`/`image`/`summary`, uniqueness `title`, validity tanggal)
- **Strategi missing values:** teks tidak diimputasi, string kosong menjadi null, baris dipertahankan dengan kolom penanda. Alasannya: 
- **Strategi outlier:** IQR + batas domain 100 karakter, windsorization pada `content_len_clip`. Jumlah outlier: 
- **Deduplikasi:** kunci `id`/`url` (hasil: ), lalu `title`+`content` (dihapus: ).
- **Keterbatasan:** sebaran artikel per tahun sangat timpang (cek tabel per tahun), periode padat hanya sekitar 2023.